# Worked examples — portfolio attribution & risk decomposition

Narrative walk-through of the engine. Built on **gs-quant** (`gs_quant.timeseries`)
for the return/risk primitives, with HAC inference, Euler risk decomposition,
Ledoit–Wolf shrinkage, Cornish–Fisher and the optimisers layered on top.

**Nothing here forecasts anything.** The regression is attribution of realised
returns; the optimisers are deterministic given the covariance; the covariance
is an estimator of present structure, not a prediction of future structure.

## 1. Data — prices to aligned excess returns

In [1]:
import numpy as np, pandas as pd
from core import data as D, diagnostics as DG, optimise as OPT
from core import risk_metrics as RM, structure as STR, tail_risk as TR
from core.attribution import factor_regression, rolling_betas
from core.conventions import annualise_cov
from core.covariance import ESTIMATORS, check_condition
from core.risk_decomp import euler_decomposition, variance_decomposition

START, END = "2018-01-01", "2025-06-30"
weights = pd.Series({"AAPL": .25, "MSFT": .20, "JPM": .20, "XOM": .20, "JNJ": .15})

prices  = D.load_prices(list(weights.index), START, END)   # yfinance, auto-adjusted
returns = D.to_returns(prices)                             # gs_quant.timeseries.returns
weights = D.align_weights(returns, weights)                # renormalise if a name dropped
port    = D.portfolio_return(returns, weights)
aligned = D.align(port, D.load_factors(START, END))        # inner join + PORT_EXCESS
names   = D.factor_names(aligned)
r_p, r_x = aligned["PORT"], aligned["PORT_EXCESS"]
print(f"{len(aligned)} aligned days, factors: {names}")
aligned.tail(3).round(4)

/Users/krishpatel/Desktop/GS_QUANT/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


/Users/krishpatel/Desktop/GS_QUANT/core/data.py:92: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  df = DataReader(name, "famafrench", start, end)[0]


/Users/krishpatel/Desktop/GS_QUANT/core/data.py:92: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  df = DataReader(name, "famafrench", start, end)[0]


1881 aligned days, factors: ['MKT', 'SMB', 'HML', 'RMW', 'MOM']


,PORT,MKT,SMB,HML,RMW,MOM,RF,PORT_EXCESS
2025-06-25,0.0046,-0.0007,-0.0117,-0.0005,-0.0014,-0.0048,0.0002,0.0044
2025-06-26,0.0074,0.0088,0.0076,0.0033,-0.0038,0.0017,0.0002,0.0072
2025-06-27,-0.0024,0.0046,-0.0028,-0.0001,0.0070,0.0045,0.0002,-0.0026


`MKT` from Ken French is already an excess return, so the portfolio side must be
excess too — `align` subtracts `RF` for exactly that reason. Regressing a raw
return on an excess factor is the classic silent error here.

## 2. Return attribution — how big is each bet, and is it real?

In [2]:
att = factor_regression(r_x, aligned[names])
print(f"R2 {att.r2:.1%} | adj R2 {att.adj_r2:.1%} | annualised alpha {att.alpha_ann:.2%} "
      f"(t = {att.alpha_tstat:.2f}) | HAC lag {att.hac_lag} | T = {att.n_obs}")
att.summary.round(3)

R2 88.5% | adj R2 88.5% | annualised alpha 4.66% (t = 1.89) | HAC lag 7 | T = 1881


,beta,t_stat,p_value,reliable
factor,,,,
MKT,0.999,84.872,0.000,True
SMB,-0.235,-11.468,0.000,True
HML,0.229,12.907,0.000,True
RMW,0.076,2.649,0.008,True
MOM,-0.031,-1.889,0.059,False


Betas come from `gs_quant.timeseries.LinearRegression`; the t-stats come from
statsmodels with a **Bartlett-kernel HAC (Newey–West)** covariance at lag
`L = floor(4·(T/100)^(2/9))`. Daily residuals are autocorrelated and
heteroskedastic, so plain OLS standard errors would be overconfident. The two
coefficient sets are asserted equal inside `factor_regression`.

`|t| ≥ 2` marks an exposure worth reading; the rest is noise dressed as a bet.

In [3]:
rb = rolling_betas(r_x, aligned[names])
rb.iloc[::63].round(2).tail(6)   # exposures drift — one full-sample beta hides that

,MKT,SMB,HML,RMW,MOM
2024-01-05,0.89,-0.22,0.12,0.22,-0.03
2024-04-08,0.78,-0.09,-0.00,0.31,-0.11
2024-07-09,0.77,-0.09,0.05,0.27,-0.14
2024-10-07,0.76,-0.13,0.21,0.22,-0.13
2025-01-07,0.83,-0.14,0.28,0.16,-0.19
2025-04-09,0.92,-0.21,0.34,0.11,-0.26


## 3. Where the risk actually lives

In [4]:
cov   = ESTIMATORS["Ledoit-Wolf"](returns.loc[aligned.index])
euler = euler_decomposition(weights, cov)
var_d = variance_decomposition(att.betas, aligned[names].cov(), att.resid.var(ddof=1))

print("variance shares sum to", round(var_d["share"].sum(), 12))
print("Euler CCR sum       =", round(euler["ccr"].sum(), 12))
print("portfolio sigma_p   =", round(euler["portfolio_vol"].iloc[0], 12))
display(var_d[["variance_contribution", "share"]].round(4))
euler[["weight", "mcr", "ccr", "pct_risk"]].round(4)

variance shares sum to 1.0
Euler CCR sum       = 0.013350493654
portfolio sigma_p   = 0.013350493654


,variance_contribution,share
factor,,
MKT,0.0002,0.8852
SMB,-0.0000,-0.0128
HML,0.0000,0.0096
RMW,-0.0000,-0.0018
MOM,0.0000,0.0046
IDIOSYNCRATIC,0.0000,0.1152


,weight,mcr,ccr,pct_risk
asset,,,,
AAPL,0.25,0.0161,0.0040,0.3015
JNJ,0.15,0.0067,0.0010,0.0756
JPM,0.20,0.0143,0.0029,0.2145
MSFT,0.20,0.0142,0.0028,0.2130
XOM,0.20,0.0130,0.0026,0.1953


Two different questions:

* **Variance decomposition** — `CCV_i = β_i (Σ_F β)_i`, summing to `βᵀΣ_Fβ`. The
  cross-terms carry factor correlation, so this is *not* just squared betas.
* **Euler volatility decomposition** — `MCR_i = (Σw)_i / σ_p`, `CCR_i = w_i·MCR_i`.
  Volatility is homogeneous of degree 1 in the weights, so by Euler's theorem the
  contributions sum **exactly** to σ_p. That identity is why risk parity is even
  a well-posed problem.

Note the gap between weight and risk share — weights are not risk.

## 4. Covariance — the input every other number depends on

In [5]:
rows = {}
for label, fn in ESTIMATORS.items():
    est = fn(returns.loc[aligned.index])
    rows[label] = {
        "ann. vol": float(np.sqrt(annualise_cov(weights.values @ np.asarray(est) @ weights.values))),
        "condition number": check_condition(est),
        "effective bets": STR.effective_bets(est),
        "top risk share": float(euler_decomposition(weights, est)["pct_risk"].max()),
    }
pd.DataFrame(rows).T.round(4)

,ann. vol,condition number,effective bets,top risk share
Sample,0.2133,9.0909,3.4571,0.3015
Ledoit-Wolf,0.2119,8.5928,3.5073,0.3015
EWMA (0.94),0.1410,9.6158,3.8095,0.3921


Ledoit–Wolf shrinks the noisy sample matrix toward a structured target with the
closed-form intensity δ*: a little bias for a large variance reduction, and a
guaranteed-invertible matrix. EWMA (λ = 0.94) weights recent structure more
heavily. Both are **estimators of present structure**, not forecasts — and every
matrix is routed through `nearest_psd`, because negative eigenvalues make risk
contributions meaningless.

## 5. Tail risk — which VaR should you believe?

In [6]:
jb = DG.jarque_bera(r_p)
print(f"Jarque-Bera p = {jb['p_value']:.3g} -> {'non-normal' if not jb['normal'] else 'normal'}; "
      f"skew {jb['skew']:.2f}, excess kurtosis {jb['excess_kurtosis']:.2f}")
TR.tail_table(r_p).round(4)

Jarque-Bera p = 0 -> non-normal; skew -0.18, excess kurtosis 12.38


,99%,95%
measure,,
VaR_historical,0.0377,0.0185
VaR_gaussian,0.0305,0.0213
VaR_cornish_fisher,0.0710,0.0186
ES_historical,0.0549,0.0320
ES_gaussian,0.0350,0.0269


Returns are fat-tailed and left-skewed, so Gaussian VaR understates the tail.
Cornish–Fisher folds skew and excess kurtosis back into the quantile —
`z_cf = z + (z²−1)S/6 + (z³−3z)K/24 − (2z³−5z)S²/36` — with no model fitted, and
it collapses back to the Gaussian number when `S = K = 0` (an invariant test).
Expected Shortfall is reported alongside because it is coherent (sub-additive)
where VaR is not.

## 6. Deterministic construction — no expected returns required

In [7]:
books = {"Current": weights,
         "Min variance": OPT.min_variance(cov, long_only=True),
         "Risk parity": OPT.risk_parity(cov),
         "Max diversification": OPT.max_diversification(cov)}
out = {}
for label, w in books.items():
    d = euler_decomposition(w, cov)
    out[label] = {"ann. vol": float(np.sqrt(annualise_cov(d["portfolio_vol"].iloc[0] ** 2))),
                  "diversification ratio": OPT.diversification_ratio(w, cov),
                  "max risk share": float(d["pct_risk"].max())}
display(pd.DataFrame(out).T.round(4))
pd.DataFrame(books).round(3)

,ann. vol,diversification ratio,max risk share
Current,0.2119,1.3517,0.3015
Min variance,0.1786,1.3286,0.6205
Risk parity,0.1947,1.3908,0.2000
Max diversification,0.1884,1.4013,0.2786


,Current,Min variance,Risk parity,Max diversification
AAPL,0.25,0.040,0.165,0.133
JNJ,0.15,0.620,0.297,0.369
JPM,0.20,0.051,0.172,0.099
MSFT,0.20,0.146,0.180,0.169
XOM,0.20,0.143,0.186,0.229


In [8]:
rp = euler_decomposition(books["Risk parity"], cov)["ccr"]
print("risk parity CCR spread (should be ~0):", float(rp.std() / rp.mean()))

risk parity CCR spread (should be ~0): 8.003711187648234e-11


Each of these needs **only Σ**. Given the covariance the solution is a
deterministic convex program — that is precisely what keeps portfolio
construction inside the no-forecasting rule. Risk parity equalises the same
Euler contributions computed in §3.

## 7. Structure — how many bets is this book really making?

In [9]:
spectrum, loadings = STR.pca(cov)
print(f"N_eff = {STR.effective_bets(cov):.2f} independent bets across {len(weights)} names")
display(spectrum.round(4))
STR.vif(aligned[names]).round(2)

N_eff = 3.51 independent bets across 5 names


,eigenvalue,explained,cumulative
component,,,
PC1,0.0009,0.5528,0.5528
PC2,0.0003,0.2090,0.7619
PC3,0.0002,0.0967,0.8586
PC4,0.0001,0.0771,0.9357
PC5,0.0001,0.0643,1.0000


factor
MKT    1.09
SMB    1.47
HML    1.52
RMW    1.43
MOM    1.20
Name: VIF, dtype: float64

`N_eff = exp(−Σ p_i ln p_i)` with `p_i = λ_i/Σλ` turns hidden concentration into
one number, bounded `1 ≤ N_eff ≤ n`. VIF flags the factor multicollinearity
(classically HML vs MOM) that destabilises individual betas.

## 8. Diagnostics — each test licenses a modelling choice

In [10]:
DG.diagnostics_table(att.resid, aligned[names]).round(4)

,statistic,p_value,conclusion,licenses
test,,,,
Jarque-Bera (residuals),223.4847,0.0000,non-normal (fat tails / skew),Cornish-Fisher VaR and ES over Gaussian VaR
"Ljung-Box (residuals, 10 lags)",11.5607,0.3155,no autocorrelation,HAC (Newey-West) errors at lag 7
ADF (MKT),-13.7405,0.0000,stationary,using this factor as a regressor
ADF (SMB),-44.1901,0.0000,stationary,using this factor as a regressor
ADF (HML),-26.6443,0.0000,stationary,using this factor as a regressor
ADF (RMW),-21.4725,0.0000,stationary,using this factor as a regressor
ADF (MOM),-28.4565,0.0000,stationary,using this factor as a regressor


## 9. The invariants — what proves the maths is right

In [11]:
sigma_p = euler["portfolio_vol"].iloc[0]
systematic = float(att.betas.values @ aligned[names].cov().values @ att.betas.values)
checks = {
    "Euler: sum(CCR) == sigma_p": np.isclose(euler["ccr"].sum(), sigma_p, atol=1e-8),
    "risk shares sum to 100%": np.isclose(euler["pct_risk"].sum(), 1.0, atol=1e-8),
    "sum(CCV) == beta' Sigma_F beta": np.isclose(
        var_d["variance_contribution"].drop("IDIOSYNCRATIC").sum(), systematic, rtol=1e-6),
    "variance shares sum to 100%": np.isclose(var_d["share"].sum(), 1.0, atol=1e-8),
    "OLS: var(fit) + var(resid) == var(y)": np.isclose(
        att.fitted.var(ddof=0) + att.resid.var(ddof=0), r_x.var(ddof=0), rtol=1e-6),
    "Sigma is PSD": np.linalg.eigvalsh(np.asarray(cov)).min() >= -1e-12,
    "1 <= N_eff <= n": 1 <= STR.effective_bets(cov) <= len(weights),
}
for k, v in checks.items():
    print(f"{'PASS' if v else 'FAIL'}  {k}")
assert all(checks.values())

PASS  Euler: sum(CCR) == sigma_p
PASS  risk shares sum to 100%
PASS  sum(CCV) == beta' Sigma_F beta
PASS  variance shares sum to 100%
PASS  OLS: var(fit) + var(resid) == var(y)
PASS  Sigma is PSD
PASS  1 <= N_eff <= n


The same guarantees are locked in for *any* input by `tests/test_invariants.py`,
alongside `tests/test_known_answers.py`, which plants known betas and alpha in
synthetic data and checks the estimator recovers them.

**Biggest limitation, stated openly:** every number is estimated on a historical
window. It is backward-looking, exposures drift, and short windows make every
estimate noisier.